# Simple RAG-LLM Action Planning

This notebook loads predictions, searches knowledge base using contributing features, and generates action plans using LLM.

In [1]:
# Task 1: Import Required Libraries
# All necessary imports for RAG, embeddings, LLM API, and data processing
import os

import json

import numpy as np

from pathlib import Path
from datetime import datetime

from typing import Dict, Any, List, Tuple

from dotenv import load_dotenv

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

from openai import OpenAI


load_dotenv()
print("✓ Imports loaded")

✓ Imports loaded


In [ ]:
# Task 2: Load Predictions and Knowledge Base
# Paths
rag_predictions_dir = Path("RAG_docs/predictions")
rag_knowledge_dir = Path("RAG_docs/knowledge")
results_action_dir = Path("RAG_docs/action_plans")
os.makedirs(results_action_dir, exist_ok=True)


# Functional loading functions
def load_predictions(predictions_dir: Path) -> List[Dict[str, Any]]:
    """Load all prediction files from the specified directory.

    Args:
        predictions_dir: Path to directory containing prediction JSON files

    Returns:
        List of prediction dictionaries with added _source_file metadata

    Raises:
        FileNotFoundError: If no JSON files found in directory
    """
    prediction_files = list(predictions_dir.glob("*.json"))
    if not prediction_files:
        raise FileNotFoundError(f"No JSON files found in {predictions_dir}")

    print(f"Found {len(prediction_files)} prediction file(s) in {predictions_dir}")

    predictions_data = []
    for prediction_file in prediction_files:
        print(f"  Loading: {prediction_file.name}")
        with open(prediction_file, "r", encoding="utf-8") as f:
            file_data = json.load(f)

        # Handle single object or array
        if isinstance(file_data, dict):
            file_data = [file_data]
        elif isinstance(file_data, list):
            pass  # Already a list
        else:
            print(
                f"    Warning: Unexpected data type in {prediction_file.name}, skipping"
            )
            continue

        # Add source file info to each prediction
        for sample in file_data:
            if isinstance(sample, dict):
                sample["_source_file"] = prediction_file.name

        predictions_data.extend(file_data)
        print(f"    Loaded {len(file_data)} prediction(s) from {prediction_file.name}")

    print(
        f"\nTotal: Loaded {len(predictions_data)} prediction(s) from {len(prediction_files)} file(s)"
    )
    return predictions_data


# Knowledge base file converters
# All converters follow the same structure: accept a file path and return list of documents
# These must be defined BEFORE load_knowledge_base() which uses them
def load_pdf_file(file_path: Path) -> List[Dict[str, Any]]:
    """Load a PDF file into document format (one dict per page with title and text).

    Args:
        file_path: Path to the PDF file.

    Returns:
        List of document dictionaries {title, text} (one per page).
    """
    loader = PyPDFLoader(str(file_path))
    docs = loader.load()
    return [
        {
            "title": f"{file_path.name} (page {doc.metadata.get('page', i + 1)})",
            "text": doc.page_content or "",
        }
        for i, doc in enumerate(docs)
    ]


def load_ddos_dataset(file_path: Path) -> List[Dict[str, Any]]:
    """Load DDoS synthetic dataset from JSON file.

    Args:
        file_path: Path to ddos-synthetic-dataset.json file

    Returns:
        List of document dictionaries (already in {title, text} format)
    """
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # DDoS dataset is already in document format (list of {title, text})
    if isinstance(data, list):
        return data
    elif isinstance(data, dict) and "title" in data and "text" in data:
        return [data]
    else:
        # Try to convert if it's a different structure
        return []


def load_knowledge_base(knowledge_dir: Path) -> List[Dict[str, Any]]:
    """Load all knowledge base files from the knowledge directory.

    Automatically discovers and loads JSON and PDF files, routing them
    to appropriate converters. JSON: by filename or standard {title, text} format.
    PDF: one document per page via PyPDFLoader.

    Args:
        knowledge_dir: Path to directory containing knowledge base files

    Returns:
        List of knowledge base documents (each dict with title and text)
    """
    knowledge_base = []

    knowledge_json = list(knowledge_dir.glob("*.json"))
    knowledge_pdf = list(knowledge_dir.glob("*.pdf"))
    if not knowledge_json and not knowledge_pdf:
        raise FileNotFoundError(
            f"No JSON or PDF files found in {knowledge_dir}"
        )

    print(f"\nLoading knowledge base files from {knowledge_dir}:")

    # File routing for JSON: map filename patterns to converter functions
    file_routes = {
        "ddos-synthetic-dataset.json": load_ddos_dataset,
    }

    # Load each JSON file
    for knowledge_file in sorted(knowledge_json):
        filename = knowledge_file.name

        if filename in file_routes:
            converter = file_routes[filename]
            docs = converter(knowledge_file)
            knowledge_base.extend(docs)
            print(f"  ✓ Loaded {len(docs)} document(s) from {filename}")
        else:
            try:
                with open(knowledge_file, "r", encoding="utf-8") as f:
                    data = json.load(f)
                if isinstance(data, list):
                    knowledge_base.extend(data)
                    print(
                        f"  ✓ Loaded {len(data)} document(s) from {filename} (standard format)"
                    )
                elif isinstance(data, dict) and "title" in data and "text" in data:
                    knowledge_base.append(data)
                    print(f"  ✓ Loaded 1 document from {filename} (standard format)")
                else:
                    print(f"  ⚠ Skipped {filename} (unknown format)")
            except Exception as e:
                print(f"  ✗ Error loading {filename}: {e}")

    # Load each PDF file (one document per page)
    for pdf_file in sorted(knowledge_pdf):
        try:
            docs = load_pdf_file(pdf_file)
            knowledge_base.extend(docs)
            print(f"  ✓ Loaded {len(docs)} document(s) from {pdf_file.name} (PDF)")
        except Exception as e:
            print(f"  ✗ Error loading {pdf_file.name}: {e}")

    print(f"\nTotal: {len(knowledge_base)} documents in knowledge base")
    return knowledge_base


# Load predictions and knowledge base using functional approach
predictions_data = load_predictions(rag_predictions_dir)
knowledge_base = load_knowledge_base(rag_knowledge_dir)

# Load attack actions and agentic features separately
# attack_options.json is in RAG_docs/ folder
attack_actions_file = Path("RAG_docs/attack_options.json")
# agentic_features.json is in RAG_docs/ folder
agentic_features_file = Path("RAG_docs/agentic_features.json")

# Load attack actions
attack_actions_data = None
if attack_actions_file.exists():
    with open(attack_actions_file, "r", encoding="utf-8") as f:
        attack_actions_data = json.load(f)
    print(f"✓ Loaded attack actions from {attack_actions_file.name}")
else:
    print(f"⚠ Attack actions file not found: {attack_actions_file}")

# Load agentic features and actions
agentic_features_data = None
if agentic_features_file.exists():
    with open(agentic_features_file, "r", encoding="utf-8") as f:
        agentic_features_data = json.load(f)
    print(f"✓ Loaded agentic features from {agentic_features_file.name}")
else:
    print(f"⚠ Agentic features file not found: {agentic_features_file}")

# Summary (useful counts)
print("\n" + "-" * 60)
print(f"Predictions to process: {len(predictions_data)} | KB documents: {len(knowledge_base)}")
n_attack = len(attack_actions_data.get("attacks", {})) if isinstance(attack_actions_data, dict) else 0
print(f"Attack types in attack_options: {n_attack}")
n_agents = len(agentic_features_data.get("agents", {})) if isinstance(agentic_features_data, dict) else 0
print(f"Agent definitions: {'yes' if n_agents > 0 else 'no'}")
print("-" * 60)


Found 1 prediction file(s) in RAG_docs\predictions
  Loading: predictions_detailed_20260209_221158.json
    Loaded 1 prediction(s) from predictions_detailed_20260209_221158.json

Total: Loaded 1 prediction(s) from 1 file(s)

Loading knowledge base files from RAG_docs\knowledge:
  ⚠ Skipped agentic_features.json (unknown format)
  ⚠ Skipped attack_options.json (unknown format)
  ✓ Loaded 121 document(s) from ddos-synthetic-dataset.json

Total: 121 documents in knowledge base
✓ Loaded attack actions from attack_options.json
⚠ Agentic features file not found: RAG_docs\knowledge\agentic_features_actions.json

------------------------------------------------------------
Predictions to process: 1 | KB documents: 121
Attack types in attack_options: 9
Agent definitions: no
------------------------------------------------------------


In [3]:
# Task 3: Create Vector Store for RAG Search
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, length_function=len
)

# Process knowledge base into documents
rag_documents = []
for idx, item in enumerate(knowledge_base):
    title = item.get("title", "")
    text = item.get("text", "")
    doc_text = f"Title: {title}\n\nContent: {text}"

    chunks = text_splitter.split_text(doc_text)
    for chunk_idx, chunk in enumerate(chunks):
        rag_documents.append(
            Document(
                page_content=chunk,
                metadata={
                    "title": title,
                    "text": text,
                    "item_index": idx,
                    "chunk_index": chunk_idx,
                },
            )
        )

print(f"Source docs: {len(knowledge_base)} → Chunks: {len(rag_documents)} (chunk_size=1000, overlap=200)")

# Create vector store using local embeddings (offline)
embed_model = "all-MiniLM-L6-v2"
print(f"Building index with {embed_model}...")
embeddings = SentenceTransformerEmbeddings(model_name=embed_model)
vector_store = FAISS.from_documents(rag_documents, embeddings)
print(f"✓ Indexed {len(rag_documents)} chunks. RAG will retrieve 10, use top 5 in prompt.")


Source docs: 121 → Chunks: 441 (chunk_size=1000, overlap=200)
Building index with all-MiniLM-L6-v2...


C:\Users\DST\AppData\Local\Temp\ipykernel_27356\4215107290.py:33: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name=embed_model)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Indexed 441 chunks. RAG will retrieve 10, use top 5 in prompt.


In [ ]:
# Task 5: LLM-Based RAG Query Generation + Feature-to-Action Mapping
# - Uses LLM to generate optimized RAG queries from sample data and SHAP contributions
# - Provides direct feature-to-action mapping from agentic_features_actions.json
# - All rule-based query generation removed - queries are now LLM-generated
def get_party_to_tier_mapping(sample: Dict[str, Any]) -> Dict[str, str]:

    """Get mapping of party names to network tiers directly from sample data.


    Extracts party names from sample's party_contributions and maps each to its network tier.

    Party names from sample: "RAN_device_party1", "Edge_device_party2", "Core_device_party3"


    Args:

        sample: Prediction sample dictionary with SHAP explanations


    Returns:

        Dictionary mapping party names to network tier names (e.g., {"RAN_device_party1": "RAN", ...})
    """

    shap_expl = sample.get("shap_explanation", {}) or {}

    party_contributions = shap_expl.get("party_contributions", {}) or {}


    party_to_tier = {}

    for party_name in party_contributions.keys():

        # Extract network tier directly from party name

        party_lower = party_name.lower()

        if party_lower.startswith("ran"):

            tier = "RAN"

        elif party_lower.startswith("edge"):

            tier = "Edge"

        elif party_lower.startswith("core"):

            tier = "Core"

        else:

            continue  # Skip if no tier match


        party_to_tier[party_name] = tier


    return party_to_tier



def get_dominant_party_info(sample: Dict[str, Any]) -> Tuple[str, str, float]:

    """Get dominant party information using party-to-tier mapping.


    Args:

        sample: Prediction sample dictionary


    Returns:

        Tuple of (party_name, network_tier, contribution_percentage)
    """

    shap_expl = sample.get("shap_explanation", {}) or {}

    dominant_agent = shap_expl.get("dominant_agent", "")

    dominant_pct = shap_expl.get("dominant_contribution_pct", 0.0) or shap_expl.get(
        "party_contributions_pct", {}
    ).get(dominant_agent, 0.0)


    if isinstance(dominant_pct, dict):

        dominant_pct = dominant_pct.get(dominant_agent, 0.0)


    # Get network tier using party-to-tier mapping (removes duplicate logic)

    party_to_tier = get_party_to_tier_mapping(sample)

    network_tier = party_to_tier.get(dominant_agent, "") if dominant_agent else ""


    return (dominant_agent, network_tier, float(dominant_pct) * 100.0)



def build_llm_based_rag_query(sample: Dict[str, Any]) -> str:
    """Generate RAG query using LLM based on sample data and SHAP contributions.

    This function uses LLM to generate an optimized RAG query from the sample's
    prediction details and SHAP feature contributions.


    Args:

        sample: Prediction sample dictionary with SHAP explanations


    Returns:

        LLM-generated RAG query string
    """

    # Extract key information from sample
    label = sample.get("predicted_label") or sample.get("true_label") or "UNKNOWN"

    confidence = sample.get("confidence", 0.0)


    # Get dominant party and network tier

    dominant_party, dominant_tier, dominant_pct = get_dominant_party_info(sample)


    # Extract SHAP information

    shap_expl = sample.get("shap_explanation", {}) or {}

    party_contributions_pct = shap_expl.get("party_contributions_pct", {}) or {}

    feat_contribs = shap_expl.get("feature_contributions", {}) or {}


    # Get party-to-tier mapping
    party_to_tier = get_party_to_tier_mapping(sample)


    # Collect top contributing features by tier
    tier_features_summary = {}
    for tier in ["RAN", "Edge", "Core"]:
        tier_features_summary[tier] = []

    # Extract top features with >0% contribution
    for party_name, feats in feat_contribs.items():
        if not isinstance(feats, dict):
            continue

        tier = party_to_tier.get(party_name, "")

        if not tier or tier not in tier_features_summary:
            continue


        # Get top 5 features for this party
        party_feat_list = []

        for feat_name, v in feats.items():

            if not isinstance(v, dict):
                continue

            pct_contrib = float(v.get("pct_contribution", 0.0) or 0.0)

            abs_shap = float(v.get("abs_shap_value", 0.0) or 0.0)

            if pct_contrib > 0.0:

                party_feat_list.append((feat_name, pct_contrib, abs_shap))

        # Sort by contribution and take top 5
        party_feat_list.sort(key=lambda x: x[1], reverse=True)
        tier_features_summary[tier].extend(
            [f"{name} ({pct:.2%})" for name, pct, _ in party_feat_list[:5]]
        )

    # Build query prompt for LLM
    query_prompt = f"""Generate a concise RAG search query to find relevant mitigation actions for a cybersecurity attack prediction.

Prediction Details:
- Attack Type: {label}
- Confidence: {confidence:.1%}
- Dominant Network Tier: {dominant_tier} (contribution: {dominant_pct:.1f}%)

Party Contributions by Network Tier:"""


    for party_name, contribution_pct in party_contributions_pct.items():

        if contribution_pct and float(contribution_pct) > 0:
            tier = party_to_tier.get(party_name, "Unknown")
            query_prompt += (
                f"\n  - {party_name} ({tier}): {float(contribution_pct):.2%}"
            )

    query_prompt += "\n\nTop Contributing Features by Network Tier:"
    for tier in ["RAN", "Edge", "Core"]:
        if tier_features_summary.get(tier):
            features_text = ", ".join(tier_features_summary[tier][:5])
            query_prompt += f"\n  - {tier}: {features_text}"

    query_prompt += f"""

Generate a search query (1-2 sentences) that will help retrieve relevant mitigation actions and strategies for this {label} attack, focusing on the {dominant_tier} network tier and the contributing features mentioned above.

Query:"""


    # Call LLM to generate query
    try:
        api_key = os.getenv("OPENAI_API_KEY")
        if api_key:
            client = OpenAI(api_key=api_key)
            response = client.chat.completions.create(
                model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
                messages=[
                    {
                        "role": "system",
                        "content": "You are a cybersecurity expert. Generate concise, effective search queries for retrieving mitigation actions.",
                    },
                    {"role": "user", "content": query_prompt},
                ],
                temperature=0.3,
                max_tokens=100,
            )
            query = response.choices[0].message.content.strip()
            return query
        else:
            # Fallback to simple query if no API key
            return f"What mitigation actions are required for {label} attack in {dominant_tier} network tier with {dominant_pct:.1f}% contribution?"
    except Exception as e:
        # Fallback to simple query on error
        return f"What mitigation actions are required for {label} attack in {dominant_tier} network tier?"



def get_actions_for_features(
    sample: Dict[str, Any], agentic_features_file: Path = None
) -> Dict[str, Any]:

    """Retrieve actions from agentic_features_actions.json based on contributing features in sample data.


    This function directly maps features from sample data to available actions in the knowledge base,

    providing a deterministic feature-to-action mapping alongside RAG retrieval.


    Args:

        sample: Prediction sample dictionary with SHAP explanations

        agentic_features_file: Path to agentic_features_actions.json (default: uses rag_knowledge_dir)


    Returns:

        Dictionary with:

        - tier_actions: Dict mapping network tier to list of available actions

        - feature_matches: Dict mapping network tier to list of matching features

        - matched_features_count: Dict mapping network tier to count of matching features

        - all_actions: List of all unique actions across all tiers
    """

    # Get file path

    if agentic_features_file is None:
        # agentic_features.json is in RAG_docs/ folder
        agentic_features_file = Path("RAG_docs/agentic_features.json")


    if not agentic_features_file.exists():

        return {

            "tier_actions": {},

            "feature_matches": {},

            "matched_features_count": {},

            "all_actions": [],

            "error": f"File not found: {agentic_features_file}",

        }


    # Load agentic features file

    with open(agentic_features_file, "r", encoding="utf-8") as f:

        agentic_data = json.load(f)


    # Extract contributing features from sample

    shap_expl = sample.get("shap_explanation", {}) or {}

    feat_contribs = shap_expl.get("feature_contributions", {}) or {}

    party_contributions_pct = shap_expl.get("party_contributions_pct", {}) or {}


    # Get party-to-tier mapping

    party_to_tier = get_party_to_tier_mapping(sample)


    # Collect all contributing features (>0%) grouped by tier

    tier_features: Dict[str, List[str]] = {"RAN": [], "Edge": [], "Core": []}


    for party_name, feats in feat_contribs.items():

        if not isinstance(feats, dict):
            continue


        tier = party_to_tier.get(party_name, "")

        if not tier or tier not in tier_features:
            continue


        for feat_name, v in feats.items():

            if not isinstance(v, dict):
                continue

            pct_contrib = float(v.get("pct_contribution", 0.0) or 0.0)

            if pct_contrib > 0.0:

                tier_features[tier].append(feat_name)


    # Get actions for each tier based on feature matches

    tier_actions = {}

    feature_matches = {}

    matched_features_count = {}

    all_actions_set = set()


    for tier in ["RAN", "Edge", "Core"]:

        if tier not in agentic_data:
            continue


        tier_data = agentic_data[tier]

        tier_feature_list = set(tier_data.get("features", []))

        tier_action_list = tier_data.get("actions", [])


        # Find matching features (features in sample that exist in tier's feature list)

        sample_features = set(tier_features.get(tier, []))

        matching_features = list(sample_features.intersection(tier_feature_list))


        # Store results

        feature_matches[tier] = matching_features

        matched_features_count[tier] = len(matching_features)


        # If there are matching features, include actions for this tier

        if matching_features:

            tier_actions[tier] = tier_action_list

            all_actions_set.update(tier_action_list)


    return {

        "tier_actions": tier_actions,  # {tier: [actions]}

        "feature_matches": feature_matches,  # {tier: [matching_feature_names]}

        "matched_features_count": matched_features_count,  # {tier: count}

        "all_actions": list(all_actions_set),  # All unique actions

        "party_contributions": {

            party: float(pct)
            for party, pct in party_contributions_pct.items()
            if pct and float(pct) > 0

        },

    }



def get_feature_action_context(
    sample: Dict[str, Any], agentic_features_file: Path = None
) -> str:

    """Get formatted feature-to-action context string for a sample.


    This is a convenience function that returns the formatted context string

    showing which actions are available for each network tier based on matching features.


    Args:

        sample: Prediction sample dictionary with SHAP explanations

        agentic_features_file: Path to agentic_features_actions.json (default: uses rag_knowledge_dir)


    Returns:

        Formatted string with feature-to-action mapping by network tier
    """

    feature_action_mapping = get_actions_for_features(sample, agentic_features_file)


    feature_action_context = ""

    if feature_action_mapping.get("tier_actions"):

        feature_action_context = (
            "\nDirect Feature-to-Action Mapping (from agentic_features_actions.json):\n"
        )

        for tier, actions in feature_action_mapping["tier_actions"].items():

            matched_count = feature_action_mapping["matched_features_count"].get(
                tier, 0
            )

            matching_features = feature_action_mapping["feature_matches"].get(tier, [])


            feature_action_context += f"\n{tier} Network Tier:\n"

            feature_action_context += f"  - Matching features: {matched_count} features with >0% contribution\n"

            if matching_features:

                # Show first 10 matching features as examples

                feature_examples = ", ".join(matching_features[:10])

                if len(matching_features) > 10:

                    feature_examples += f", ... ({len(matching_features) - 10} more)"

                feature_action_context += (
                    f"  - Example matching features: {feature_examples}\n"
                )

            feature_action_context += f"  - Available actions: {', '.join(actions)}\n"


        all_actions = feature_action_mapping.get("all_actions", [])

        if all_actions:

            feature_action_context += (
                f"\nAll available actions across tiers: {', '.join(all_actions)}\n"
            )

    else:

        feature_action_context = (
            "\nNo matching features found in agentic_features_actions.json\n"
        )


    return feature_action_context

In [5]:
# 5. Create LLM Prompt with Party Evidence and RAG Context
# Updated create_prompt function for dynamic party system
# Replace the existing create_prompt function in cell 5 of RAG_LLM_action_plan.ipynb


def create_prompt(sample_data, rag_results):
    """Create prompt with prediction details, party evidence, contributing features, and RAG context (top 5 docs).

    Uses dynamic party system with network tiers (RAN, Edge, Core) and extracts party contributions
    dynamically from the sample data.
    """
    sample_id = sample_data.get("sample_id", 0)
    predicted_label = sample_data.get("predicted_label", "UNKNOWN")
    confidence = sample_data.get("confidence", 0.0)

    # Extract party contributions dynamically
    shap_explanation = sample_data.get("shap_explanation", {}) or {}

    # Get dominant party information using dynamic system
    dominant_party, dominant_tier, dominant_pct = get_dominant_party_info(sample_data)

    # Get party-to-tier mapping
    party_to_tier = get_party_to_tier_mapping(sample_data)

    # Build attack actions context from attack_options.json
    attack_actions_context = ""
    if attack_actions_data and "attacks" in attack_actions_data:
        attack_type = predicted_label.upper()
        if attack_type in attack_actions_data["attacks"]:
            recommended_actions = attack_actions_data["attacks"][attack_type]
            attack_actions_context = (
                f"\n\nAttack-Specific Recommended Actions (from attack_options.json):\n"
            )
            attack_actions_context += f"For {predicted_label} attack, recommended actions: {', '.join(recommended_actions)}\n"
        else:
            attack_actions_context = f"\n\nAttack-Specific Actions: No specific recommendations found for {predicted_label} attack type.\n"

    # Build agentic features context from agentic_features_actions.json
    agentic_context = ""
    if agentic_features_data:
        agentic_context = "\n\nAgentic Features and Actions by Network Tier (from agentic_features_actions.json):\n"
        for tier in ["RAN", "Edge", "Core"]:
            if tier in agentic_features_data:
                tier_data = agentic_features_data[tier]
                features = tier_data.get("features", [])
                actions = tier_data.get("actions", [])
                agentic_context += f"\n{tier} Network Tier:\n"
                agentic_context += (
                    f"  - Available features: {len(features)} total features\n"
                )
                agentic_context += f"  - Available actions: {', '.join(actions)}\n"

    # Use top 5 RAG results for LLM
    top_5_results = rag_results[:5]

    # Build RAG context
    rag_context = ""
    if top_5_results:
        rag_context = "\n\nKnowledge Base Context (from RAG search):\n"
        for idx, result in enumerate(top_5_results, 1):
            rag_context += f"\n[{idx}] {result['title']}\n"
            rag_context += f"{result['text'][:1000]}\n"
    else:
        rag_context = "\n\nKnowledge Base: No relevant documents found from RAG search."

    # Build network tier context for prompt
    network_tier_info = ""
    if dominant_tier:
        network_tier_info = f"\n- Dominant network tier: {dominant_tier} (contribution: {dominant_pct:.1f}%)"
        if dominant_party:
            network_tier_info += f"\n- Dominant party: {dominant_party}"
    prompt = f"""You are a cybersecurity decision-making agent specialized in attack response orchestration.
        Your role is NOT to invent mitigations, but to SELECT and ASSIGN actions from a predefined
        action set using explainability signals and agentic features.

        =====================
        INPUT CONTEXT
        =====================

        Prediction summary:
        - sample_id: {sample_id}
        - predicted_label: {predicted_label}
        - confidence: {confidence:.1%}

        Explainability & agentic evidence:
        - Party-level contributions and dominance:
        {network_tier_info}

        - Feature-level evidence (SHAP / statistics / signals):
        {str(sample_data)}

        Allowed actions (STRICT CONSTRAINT):
        {attack_actions_context}
        • Only actions listed above are allowed.
        • Do NOT invent, rename, generalize, or merge actions.

        Agentic decision signals:
        {agentic_context}

        Retrieved knowledge (RAG – optional support, may be empty):
        {rag_context}

        =====================
        TASK
        =====================

        Using ONLY the information above, generate an agent-ready action plan that:

        1) Interprets how {predicted_label} manifests across network tiers (RAN, Edge, Core).
        2) Identifies which evidence party MUST trigger mitigation first.
        3) Selects actions ONLY from the provided Allowed actions list.
        4) Assigns each action to the MOST appropriate executing party and network tier.
        5) Provides explicit, evidence-grounded reasoning for EACH action.
        6) Adapts aggressiveness and execution priority based on confidence ({confidence:.1%}).
        7) If no allowed action is suitable, return empty action lists.

        =====================
        OUTPUT FORMAT (STRICT)
        =====================

        Return a VALID JSON object ONLY:

        {{
          "threat_level": "Critical|High|Medium|Low",

          "all_actions": [
            "action_name_1",
            "action_name_2",
            "action_name_3"
          ],

          "primary_actions": [
            {{
              "action": "EXACT action name from Allowed actions to be taken ",
              "network_tier": "RAN|Edge|Core",
              "party_evidence_type": "type of evidence this party observed",
              "reasoning": "clear explanation linking evidence + agentic signals to this action"
            }}
          ],

          "supporting_actions": [
            {{
              "action": "EXACT action name from Allowed actions to be taken",
              "network_tier": "RAN|Edge|Core",
              "party_evidence_type": "type of evidence this party observed",
              "reasoning": "why this action supports or complements the primary action"
            }}
          ],
          "overall_reasoning": "Concise summary explaining party prioritization, tier ordering, and action selection logic",
          "execution_priority": "Immediate|High|Standard|Low",
          "knowledge_sources_used": [
            "allowed_actions_context",
            "attack_actions_context"
          ]
        }}

        =====================
        HARD RULES (DO NOT VIOLATE)
        =====================

        - Do NOT output text outside the JSON.
        - Do NOT generate actions not listed in Allowed actions.
        - The "all_actions" list MUST be the union of primary_actions and supporting_actions.
        - Do NOT alter action or party names.
        - Every action MUST include explicit reasoning tied to evidence or agentic rules.
        - Prefer dominant party and tier for primary actions unless contradicted by evidence.
        - If RAG context is empty, rely ONLY on explainability and agentic context.
        """


    return prompt

print("=" * 60)
print("TASK 5: Query generation and prompt helpers")
print("=" * 60)
print("  Defined: get_party_to_tier_mapping, build_llm_based_rag_query")
print("  create_prompt (with RAG, top 5 docs)")
print("=" * 60)


TASK 5: Query generation and prompt helpers
  Defined: get_party_to_tier_mapping, build_llm_based_rag_query
  create_prompt (with RAG, top 5 docs)


In [6]:
def create_prompt_without_RAG(sample_data):
    """Create prompt with prediction details, party evidence, contributing features, WITHOUT RAG context."""
    sample_id = sample_data.get("sample_id", 0)
    predicted_label = sample_data.get("predicted_label", "UNKNOWN")
    confidence = sample_data.get("confidence", 0.0)
    
    prompt = f"""You are a cybersecurity expert responsible for selecting mitigation actions
      based on attack predictions and domain knowledge.

      Your objective is to decide WHAT actions should be taken and WHY,
      strictly using a predefined set of allowed actions.

      =====================
      INPUT
      =====================

      Prediction summary:
      - sample_id: {sample_id}
      - predicted_label: {predicted_label}
      - confidence: {confidence:.1%}

     
      =====================
      TASK
      =====================

      Using ONLY the information above:

      1) Assess the severity of the predicted attack ({predicted_label})
        and assign an appropriate threat level.
      2) Select all relevant mitigation actions within 2 or 3 words
      3) Ensure selected actions are appropriate for the predicted attack type.
      4) Adapt action selection and urgency based on confidence ({confidence:.1%}).
      5) If no action is applicable, return an empty action list.

      =====================
      OUTPUT (STRICT JSON ONLY)
      =====================

      {{
        "threat_level": "Critical|High|Medium|Low",
        "all_actions": [
            "action_name_1",
            "action_name_2",
            "action_name_3"
        ],
        "primary_actions": [
            {{
              "action": "any action name  to be taken",
              "network_tier": "RAN|Edge|Core",
              "reasoning": "clear explanation linking evidence + agentic signals to this action"
            }}
          ],
        "overall_reasoning": "Brief explanation describing why these actions were selected and how confidence influenced the decision",
        "execution_priority": "Immediate|High|Standard|Low",
        "knowledge_sources_used": [
          "publicly_available_online_sources"
        ]
      }}

      =====================
      HARD RULES
      =====================

      - Do NOT output text outside the JSON.
      - Do NOT invent, rename, or generalize actions.
      - Use EXACT action names from Allowed actions.
      - If no allowed action applies, return `"actions": []`.
      - Keep reasoning concise and decision-focused.
      """
    return prompt

print("=" * 60)
print("TASK 5 (cont.): create_prompt_without_RAG defined")
print("=" * 60)


TASK 5 (cont.): create_prompt_without_RAG defined


In [7]:
# Task 8: RAG Knowledge Base Search (Vector Similarity)
def search_rag_knowledge(query, top_k=10):
    """Search knowledge base using vector similarity search.
    
    Uses the provided query (generated by LLM) to search documents via vector similarity.
    No LLM-based ranking - just pure vector similarity search.
    
    Args:
        query: Search query string (generated by build_llm_based_rag_query)
        top_k: Number of results to return
    
    Returns:
        List of formatted results with title, text, and similarity score
    """
    if vector_store is None or not knowledge_base:
        return []
    
    # Perform vector similarity search
    vector_results = vector_store.similarity_search_with_score(query, k=top_k)
    
    # Build formatted results
    formatted_results = []
    seen_titles = set()
    
    for doc, score in vector_results:
        title = doc.metadata.get("title", "Unknown")
        text = doc.metadata.get("text", "")
        
        # Avoid duplicates - use first occurrence of each title
        if title not in seen_titles:
            seen_titles.add(title)
            
            # Get full text from knowledge base if not in metadata
            full_text = text if text else ""
            if not full_text or len(full_text) < 100:
                # Try to find in knowledge_base
                for kb_item in knowledge_base:
                    if kb_item.get("title") == title:
                        full_text = kb_item.get("text", text)
                        break
            
            # Calculate similarity score (lower distance = higher similarity)
            similarity_score = 1 / (1 + float(score)) if score > 0 else 1.0
            
            formatted_results.append({
                "title": title,
                "text": full_text,
                "score": similarity_score
            })
    
    return formatted_results

# Task 9: Save Comparison File (With-RAG vs Without-RAG)
def save_comparison_file(
    sample,
    query,
    rag_results,
    llm_response_with_rag,
    llm_response_without_rag,
    results_action_dir,
    with_rag_file,
    without_rag_file,
):
    """Save a comparison JSON file containing both with-RAG and without-RAG action plans.

    Args:
        sample: Prediction sample data
        query: RAG query used
        rag_results: RAG search results
        llm_response_with_rag: LLM response with RAG context
        llm_response_without_rag: LLM response without RAG context
        results_action_dir: Directory to save results
        with_rag_file: Path to the with-RAG action plan file
        without_rag_file: Path to the without-RAG action plan file
    """
    comparison = {
        "sample_id": sample.get("sample_id"),
        "prediction": {
            "predicted_label": sample.get("predicted_label"),
            "confidence": float(sample.get("confidence", 0.0)),
        },
        "comparison_metadata": {
            "compared_at": datetime.now().isoformat(),
            "with_rag_file": with_rag_file.name if with_rag_file else None,
            "without_rag_file": without_rag_file.name if without_rag_file else None,
            "rag_query": query,
            "rag_documents_found": len(rag_results),
        },
        "rag_context": {
            "documents_used": len(rag_results),
            "top_documents": [
                {
                    "title": r["title"],
                    "similarity_score": float(r["score"]),
                    "text_preview": (
                        r["text"][:200] + "..." if len(r["text"]) > 200 else r["text"]
                    ),
                }
                for r in rag_results[:5]
            ],
        },
        "action_plans": {
            "with_rag": {
                "threat_level": (
                    llm_response_with_rag.get("threat_level")
                    if llm_response_with_rag
                    else None
                ),
                "execution_priority": (
                    llm_response_with_rag.get("execution_priority")
                    if llm_response_with_rag
                    else None
                ),
                "primary_actions": (
                    llm_response_with_rag.get("primary_actions", [])
                    if llm_response_with_rag
                    else []
                ),
                "supporting_actions": (
                    llm_response_with_rag.get("supporting_actions", [])
                    if llm_response_with_rag
                    else []
                ),
               
                "overall_reasoning": (
                    (llm_response_with_rag.get("overall_reasoning") or llm_response_with_rag.get("reasoning", ""))
                    if llm_response_with_rag
                    else ""
                ),
                "knowledge_sources_used": (
                    llm_response_with_rag.get("knowledge_sources_used", [])
                    if llm_response_with_rag
                    else []
                ),
            },
            "without_rag": {
                "threat_level": (
                    llm_response_without_rag.get("threat_level")
                    if llm_response_without_rag
                    else None
                ),
                "execution_priority": (
                    llm_response_without_rag.get("execution_priority")
                    if llm_response_without_rag
                    else None
                ),
                "primary_actions": (
                    llm_response_without_rag.get("primary_actions", [])
                    if llm_response_without_rag
                    else []
                ),
                "supporting_actions": (
                    llm_response_without_rag.get("supporting_actions", [])
                    if llm_response_without_rag
                    else []
                ),
               
                "overall_reasoning": (
                    (llm_response_without_rag.get("overall_reasoning") or llm_response_without_rag.get("reasoning", ""))
                    if llm_response_without_rag
                    else ""
                ),
                "knowledge_sources_used": (
                    llm_response_without_rag.get("knowledge_sources_used", [])
                    if llm_response_without_rag
                    else []
                ),
            },
        },
        "differences": {
            "threat_level_different": (
                llm_response_with_rag.get("threat_level")
                if llm_response_with_rag
                else None
            )
            != (
                llm_response_without_rag.get("threat_level")
                if llm_response_without_rag
                else None
            ),
            "priority_different": (
                llm_response_with_rag.get("execution_priority")
                if llm_response_with_rag
                else None
            )
            != (
                llm_response_without_rag.get("execution_priority")
                if llm_response_without_rag
                else None
            ),
            "primary_actions_different": (
                llm_response_with_rag.get("primary_actions", [])
                if llm_response_with_rag
                else []
            )
            != (
                llm_response_without_rag.get("primary_actions", [])
                if llm_response_without_rag
                else []
            ),
            "num_primary_actions_with_rag": (
                len(llm_response_with_rag.get("primary_actions", []))
                if llm_response_with_rag
                else 0
            ),
            "num_primary_actions_without_rag": (
                len(llm_response_without_rag.get("primary_actions", []))
                if llm_response_without_rag
                else 0
            ),
            "overall_reasoning_different": (
                (llm_response_with_rag.get("overall_reasoning", "") or llm_response_with_rag.get("reasoning", ""))
                if llm_response_with_rag
                else ""
            )
            != (
                (llm_response_without_rag.get("overall_reasoning", "") or llm_response_without_rag.get("reasoning", ""))
                if llm_response_without_rag
                else ""
            ),
        },
    }

    # Convert any numpy types to native Python types
    comparison = convert_to_json_serializable(comparison)

    # Create filename with timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    comparison_file = (
        results_action_dir
        / f"comparison_sample_{sample.get('sample_id')}_{timestamp}.json"
    )

    with open(comparison_file, "w", encoding="utf-8") as f:
        json.dump(comparison, f, indent=2, ensure_ascii=False)

    return comparison_file

In [8]:
# 7. Save Action Plan
def convert_to_json_serializable(obj):
    """Convert numpy types and other non-serializable types to JSON-compatible types."""
    import numpy as np

    if isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {key: convert_to_json_serializable(value) for key, value in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_to_json_serializable(item) for item in obj]
    else:
        return obj


def save_action_plan(sample, query, rag_results, llm_response, results_action_dir):
    """Save action plan in the specified format."""
    result = {
        "sample_id": sample.get("sample_id"),
        "prediction": {
            "predicted_label": sample.get("predicted_label"),
            "confidence": float(sample.get("confidence", 0.0)),
        },
        "rag_info": {
            "documents_used": len(rag_results),
            "search_method": "vector_similarity",
            "search_queries": [query] if query else [],
            "top_results": [
                {
                    "title": r["title"],
                    "text": r["text"],
                    "similarity_score": float(r["score"]),
                }
                for r in rag_results[:5]
            ],
        },
        "llm_action_plan": llm_response,
        "processed_at": datetime.now().isoformat(),
    }

    # Convert any numpy types to native Python types
    result = convert_to_json_serializable(result)

    # Create filename with timestamp (include microseconds to avoid collisions)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    output_file = (
        results_action_dir
        / f"action_plan_sample_{sample.get('sample_id')}_{timestamp}.json"
    )

    try:
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        # Verify file was created
        if not output_file.exists():
            raise FileNotFoundError(f"File was not created: {output_file}")
        return output_file
    except Exception as e:
        print(f"Error in save_action_plan: {e}")
        raise

print("=" * 60)
print("TASK 8 & 9: RAG search and save helpers")
print("=" * 60)
print("  Defined: search_rag_knowledge (top_k=10, returns top 5 to LLM)")
print("  save_comparison_file, save_action_plan")
print("=" * 60)


TASK 8 & 9: RAG search and save helpers
  Defined: search_rag_knowledge (top_k=10, returns top 5 to LLM)
  save_comparison_file, save_action_plan


In [9]:
# Task 11: Call LLM API and Process All Predictions
def call_llm_api(prompt):
    """Call LLM API and return JSON response."""
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("ERROR: OPENAI_API_KEY not set")
        return None

    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        messages=[
            {
                "role": "system",
                "content": "You are a cybersecurity expert. Return only valid JSON.",
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0.3,
    )

    response_text = response.choices[0].message.content.strip()
    try:
        start = response_text.find("{")
        end = response_text.rfind("}") + 1
        if start >= 0 and end > start:
            return json.loads(response_text[start:end])
    except Exception as e:
        print(f"Warning: Could not parse JSON: {e}")

    return {
        "threat_level": "Unknown",
        "primary_actions": ["Unable to parse response"],
        "supporting_actions": [],
        "reasoning": response_text[:200],
        "execution_priority": "Standard",
        "knowledge_sources_used": [],
    }


# Process ALL predictions (including BENIGN) with RAG and attack-focused prompts
samples_processed = 0
for sample in predictions_data:
    print("=" * 80)
    print(f"Processing Sample ID: {sample.get('sample_id')}")
    predicted_label = sample.get("predicted_label", "UNKNOWN")
    print(f"Predicted Label: {predicted_label}")
    print(f"Confidence: {sample.get('confidence', 0):.1%}")

    # Process ALL predictions with RAG (including BENIGN)
    # Create LLM-based RAG query from sample data and SHAP contributions
    query = build_llm_based_rag_query(sample)
    print("\n--- RAG query sent to vector search ---")
    print(query)
    print("")

    # Search knowledge base (retrieve 10, use top 5 for LLM)
    rag_results = search_rag_knowledge(query, top_k=10)
    top5 = rag_results[:5]
    print(f"--- Retrieved {len(rag_results)} docs; top 5 used in prompt ---")
    for idx, result in enumerate(top5, 1):
        snippet_len = 350
        text_preview = (result.get("text", "") or "")[:snippet_len]
        if len(result.get("text", "") or "") > snippet_len:
            text_preview += "..."
        print(f"\n  [{idx}] {result.get('title', 'Unknown')} (similarity: {result.get('score', 0):.2%})")
        print(f"      {text_preview}")
    if not top5:
        print("  (no documents retrieved)")
    print("")

    # Create prompt with top 5 RAG results (for all predictions including BENIGN)
    prompt = create_prompt(sample, rag_results)
    prompt_without_rag = create_prompt_without_RAG(sample)

    print("--- PROMPT SENT TO LLM (with RAG) ---")
    print(prompt)
    print("--- END PROMPT ---")

    # Call LLM with RAG
    print("\nCalling LLM API with RAG...")
    llm_response_with_rag = call_llm_api(prompt)

    # Call LLM without RAG
    print("\nCalling LLM API without RAG...")
    llm_response_without_rag = call_llm_api(prompt_without_rag)

    # Save with-RAG result
    output_file_with_rag = None
    if llm_response_with_rag:
        r = llm_response_with_rag
        print("\n--- LLM response (with RAG) ---")
        print(f"  Threat level: {r.get('threat_level')}")
        print(f"  Execution priority: {r.get('execution_priority')}")
        print(f"  Primary actions: {r.get('primary_actions', [])}")
        reasoning = (r.get('overall_reasoning') or "")[:500]
        if len(r.get('overall_reasoning') or "") > 500:
            reasoning += "..."
        print(f"  Overall reasoning: {reasoning}")

        # Save result using reusable function
        try:
            output_file_with_rag = save_action_plan(
                sample, query, rag_results, llm_response_with_rag, results_action_dir
            )
            print(f"✓ Saved with-RAG result to {output_file_with_rag.name}")
            print(f"  Full path: {output_file_with_rag}")
        except Exception as e:
            print(f"✗ Error saving with-RAG result: {e}")
            output_file_with_rag = None
    else:
        print("✗ Failed to get LLM response with RAG")
        output_file_with_rag = None

    # Save without-RAG result
    without_rag_path = None
    if llm_response_without_rag:
        r = llm_response_without_rag
        print("\n--- LLM response (without RAG) ---")
        print(f"  Threat level: {r.get('threat_level')}")
        print(f"  Execution priority: {r.get('execution_priority')}")
        print(f"  Primary actions: {r.get('primary_actions', [])}")

        # Save without-RAG result (pass empty rag_results)
        output_file_without_rag = save_action_plan(
            sample, query, [], llm_response_without_rag, results_action_dir
        )
        # Rename to indicate it's without RAG
        without_rag_path = (
            output_file_without_rag.parent
            / output_file_without_rag.name.replace("action_plan_", "action_plan_noRAG_")
        )
        output_file_without_rag.rename(without_rag_path)
        print(f"✓ Saved without-RAG result to {without_rag_path.name}")
    else:
        print("✗ Failed to get LLM response without RAG")
        output_file_without_rag = None

    # Save comparison file if both responses exist
    if llm_response_with_rag and llm_response_without_rag:
        comparison_file = save_comparison_file(
            sample,
            query,
            rag_results,
            llm_response_with_rag,
            llm_response_without_rag,
            results_action_dir,
            output_file_with_rag,
            without_rag_path,
        )
        print(f"✓ Saved comparison file to {comparison_file.name}")

    samples_processed += 1
    print("=" * 80)

print("")
print("=" * 60)
print("COMPLETE")
print("=" * 60)
print(f"  Processed {samples_processed} sample(s)")
print(f"  Action plans saved to: {results_action_dir}")
print("=" * 60)

Processing Sample ID: 0
Predicted Label: DDOS
Confidence: 100.0%

--- RAG query sent to vector search ---
"DDOS mitigation strategies for RAN network tier focusing on high-risk features: udps.srcdst_rst_packet_count, udps.srcdst_syn_packet_count, udps.syn_packet_count, udps.rst_packet_count, udps.fin_packet_count."

--- Retrieved 8 docs; top 5 used in prompt ---

  [1] How can cloud providers use multi-tiered DDoS protection strategies to enhance security? (similarity: 61.76%)
      Multi-tiered DDoS protection strategies involve implementing several layers of defense to effectively mitigate distributed denial-of-service (DDoS) attacks. Cloud providers can enhance security using the following approaches:

1. **Network Layer Protection**:
   - **Traffic Filtering**: Utilize firewalls and intrusion detection/prevention systems (...

  [2] What strategies can be employed to defend against DDoS attacks on cloud storage services? (similarity: 61.65%)
      To defend against DDoS (Distribute